# Embedding-Only Retriever with Query Expansion

Clean implementation using:
- **Qwen Embeddings + BM25** for retrieval
- **Query Expansion** for conversation history context
- **No Re-ranker** (embeddings are more accurate for your use case)

In [ ]:
import yaml
import json
import os
import pickle
from typing import List, Dict, Optional
import numpy as np
import faiss
import boto3
import uuid
from datetime import datetime

with open('config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("✓ Config loaded")

In [ ]:
class EmbeddingRetriever:
    """
    Production retriever using embeddings only.
    
    Features:
    - Qwen embeddings + BM25 hybrid search
    - Query expansion for conversation history
    - Session management
    - Entitlement filtering
    """
    
    def __init__(self, config: dict):
        self.config = config
        
        # Initialize embedding client
        self.embedding_endpoint_name = config['models']['embedding']['endpoint_name']
        embedding_creds = config['models']['embedding']['credentials']
        self.embedding_client = boto3.client(
            'sagemaker-runtime',
            region_name=embedding_creds['region'],
            aws_access_key_id=embedding_creds['accessKeyId'],
            aws_secret_access_key=embedding_creds['secretAccessKey'],
            aws_session_token=embedding_creds['sessionToken']
        )
        print("✓ Embedding client initialized")
        
        # Session storage
        self.sessions = {}
        
        # Load indexes
        self.load_indexes()
    
    def load_indexes(self):
        """Load FAISS and BM25 indexes"""
        faiss_path = os.path.join(self.config['storage']['faiss_index'], 'faiss.index')
        self.faiss_index = faiss.read_index(faiss_path)
        print("✓ FAISS index loaded")
        
        bm25_path = os.path.join(self.config['storage']['bm25_index'], 'bm25.pkl')
        with open(bm25_path, 'rb') as f:
            self.bm25_index = pickle.load(f)
        print("✓ BM25 index loaded")
        
        metadata_path = os.path.join(self.config['storage']['faiss_index'], 'chunk_metadata.json')
        with open(metadata_path, 'r') as f:
            self.chunks = json.load(f)
        print(f"✓ Loaded {len(self.chunks)} chunks")
    
    def get_embedding(self, text: str) -> np.ndarray:
        """Get embedding from Qwen endpoint"""
        params = {"inputs": [text], "encoding_format": "float"}
        response = self.embedding_client.invoke_endpoint(
            EndpointName=self.embedding_endpoint_name,
            ContentType='application/json',
            Body=json.dumps(params)
        )
        raw_bytes = response['Body'].read()
        output_data = json.loads(raw_bytes.decode())
        return np.array(output_data[0], dtype='float32')
    
    # =========================================================================
    # QUERY EXPANSION FOR CONVERSATION HISTORY
    # =========================================================================
    
    def expand_query(self, query: str, history: List[Dict], 
                     max_history: int = 3,
                     expansion_mode: str = 'prepend') -> str:
        """
        Expand query with conversation history context.
        
        Args:
            query: Current user query
            history: List of previous turns [{'query': '...', 'documents_found': [...]}]
            max_history: Maximum number of history turns to include
            expansion_mode: 'prepend', 'append', or 'natural'
        
        Returns:
            Expanded query string
        """
        if not history:
            return query
        
        # Get recent history
        recent = history[-max_history:]
        
        # Extract previous queries
        prev_queries = [h.get('query', '') for h in recent if h.get('query')]
        
        if not prev_queries:
            return query
        
        if expansion_mode == 'prepend':
            # Prepend history queries
            # "cancel booking" + "refund policy" + "What documents?" 
            # → "cancel booking. refund policy. What documents?"
            context = ". ".join(prev_queries)
            return f"{context}. {query}"
        
        elif expansion_mode == 'append':
            # Append history as context
            context = ". ".join(prev_queries)
            return f"{query} (context: {context})"
        
        elif expansion_mode == 'natural':
            # Natural language context
            # → "Regarding cancel booking and refund policy: What documents?"
            context = " and ".join(prev_queries[-2:]).lower()
            return f"Regarding {context}: {query}"
        
        elif expansion_mode == 'keywords':
            # Extract key terms only
            # → "cancel booking refund policy What documents?"
            context = " ".join(prev_queries)
            return f"{context} {query}"
        
        else:
            return query
    
    # =========================================================================
    # HYBRID SEARCH (EMBEDDING + BM25)
    # =========================================================================
    
    def hybrid_search(self, query: str, entitlement: str, 
                      org_id: str = None, tags: List[str] = None,
                      top_k: int = 10) -> List[Dict]:
        """
        Hybrid search combining vector similarity and BM25.
        """
        # Get query embedding
        query_embedding = self.get_embedding(query)
        query_embedding = query_embedding.reshape(1, -1).astype('float32')
        faiss.normalize_L2(query_embedding)
        
        # Search more candidates for filtering
        initial_top_k = min(top_k * 10, len(self.chunks))
        
        # Vector search (FAISS)
        vector_scores, vector_indices = self.faiss_index.search(query_embedding, initial_top_k)
        vector_scores = vector_scores[0]
        vector_indices = vector_indices[0]
        
        # BM25 search
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25_index.get_scores(tokenized_query)
        
        # Normalize scores
        def normalize(scores):
            min_s, max_s = scores.min(), scores.max()
            if max_s - min_s < 1e-10:
                return np.zeros_like(scores)
            return (scores - min_s) / (max_s - min_s)
        
        vector_scores_norm = normalize(vector_scores)
        bm25_scores_norm = normalize(bm25_scores)
        
        # Get weights from config
        vector_weight = self.config['retrieval']['hybrid']['vector_weight']
        bm25_weight = self.config['retrieval']['hybrid']['bm25_weight']
        
        # Compute hybrid scores
        hybrid_scores = {}
        
        for idx, score in zip(vector_indices, vector_scores_norm):
            hybrid_scores[idx] = score * vector_weight
        
        for idx, score in enumerate(bm25_scores_norm):
            if idx in hybrid_scores:
                hybrid_scores[idx] += score * bm25_weight
            else:
                hybrid_scores[idx] = score * bm25_weight
        
        # Sort by hybrid score
        sorted_indices = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)
        
        # Filter by entitlement and collect results
        results = []
        
        for idx, score in sorted_indices:
            chunk = self.chunks[idx].copy()
            
            # Check entitlement
            chunk_entitlements = chunk['entitlement']
            if isinstance(chunk_entitlements, str):
                chunk_entitlements = [chunk_entitlements]
            
            has_access = 'universal' in chunk_entitlements or entitlement in chunk_entitlements
            if not has_access:
                continue
            
            # Check org_id
            if org_id and chunk.get('orgId') != org_id:
                continue
            
            # Check tags
            if tags and not any(t in chunk.get('metadata', {}).get('tags', []) for t in tags):
                continue
            
            chunk['score'] = float(score)
            results.append(chunk)
            
            if len(results) >= top_k:
                break
        
        return results
    
    # =========================================================================
    # MAIN QUERY METHOD
    # =========================================================================
    
    def query(self, query: str, entitlement: str,
              org_id: str = None, tags: List[str] = None,
              top_k: int = 5,
              conversation_history: List[Dict] = None,
              expansion_mode: str = 'prepend',
              max_history: int = 3,
              verbose: bool = False) -> Dict:
        """
        Query with optional conversation history expansion.
        
        Args:
            query: User's search query
            entitlement: User's access level
            org_id: Organization filter
            tags: Tag filters
            top_k: Number of results to return
            conversation_history: Previous conversation turns
            expansion_mode: 'prepend', 'append', 'natural', or 'keywords'
            max_history: Number of history turns to include
            verbose: Print debug information
        
        Returns:
            Dict with query results
        """
        # Expand query with history
        expanded_query = self.expand_query(
            query=query,
            history=conversation_history,
            max_history=max_history,
            expansion_mode=expansion_mode
        )
        
        if verbose:
            print(f"Original query: {query}")
            if conversation_history:
                print(f"History turns: {len(conversation_history)}")
                print(f"Expanded query: {expanded_query}")
        
        # Search with expanded query
        results = self.hybrid_search(
            query=expanded_query,
            entitlement=entitlement,
            org_id=org_id,
            tags=tags,
            top_k=top_k * 2  # Get extra for deduplication
        )
        
        if verbose:
            print(f"\nSearch results:")
            for i, r in enumerate(results[:5]):
                print(f"  {i+1}. {r['title']} (score: {r['score']:.4f})")
        
        # Deduplicate by doc_id
        seen_docs = set()
        documents = []
        
        for chunk in results:
            doc_id = chunk['doc_id']
            if doc_id not in seen_docs:
                seen_docs.add(doc_id)
                documents.append({
                    'document_name': chunk['title'],
                    'doc_id': doc_id,
                    'score': chunk['score'],
                    'content_preview': chunk['content'][:200] + '...' if len(chunk['content']) > 200 else chunk['content']
                })
                
                if len(documents) >= top_k:
                    break
        
        return {
            'query': query,
            'expanded_query': expanded_query if conversation_history else None,
            'documents': documents,
            'history_turns_used': len(conversation_history) if conversation_history else 0
        }
    
    # =========================================================================
    # SESSION MANAGEMENT
    # =========================================================================
    
    def create_session(self, user_id: str, entitlement: str, 
                       org_id: str = None) -> str:
        """Create a new conversation session"""
        session_id = str(uuid.uuid4())
        self.sessions[session_id] = {
            'session_id': session_id,
            'user_id': user_id,
            'entitlement': entitlement,
            'org_id': org_id,
            'query_history': [],
            'created_at': datetime.now().isoformat()
        }
        print(f"✓ Created session: {session_id}")
        return session_id
    
    def query_with_session(self, session_id: str, query: str,
                           tags: List[str] = None, top_k: int = 5,
                           expansion_mode: str = 'prepend',
                           max_history: int = 3,
                           verbose: bool = False) -> Dict:
        """
        Query using session for automatic history tracking.
        """
        session = self.sessions.get(session_id)
        if not session:
            raise ValueError(f"Session not found: {session_id}")
        
        # Get conversation history
        history = session['query_history']
        
        # Perform query with history
        result = self.query(
            query=query,
            entitlement=session['entitlement'],
            org_id=session['org_id'],
            tags=tags,
            top_k=top_k,
            conversation_history=history,
            expansion_mode=expansion_mode,
            max_history=max_history,
            verbose=verbose
        )
        
        # Store this query in history
        session['query_history'].append({
            'query': query,
            'timestamp': datetime.now().isoformat(),
            'documents_found': [d['document_name'] for d in result['documents']]
        })
        
        result['session_id'] = session_id
        return result
    
    def get_session_history(self, session_id: str) -> List[Dict]:
        """Get conversation history for a session"""
        session = self.sessions.get(session_id)
        return session['query_history'] if session else []
    
    def clear_session(self, session_id: str):
        """Clear a session"""
        if session_id in self.sessions:
            del self.sessions[session_id]
            print(f"✓ Session {session_id} cleared")

print("✓ EmbeddingRetriever class defined")

## Initialize Retriever

In [ ]:
retriever = EmbeddingRetriever(config=config)

## Test 1: Basic Query (No History)

In [ ]:
print("="*60)
print("TEST 1: Basic Query (No History)")
print("="*60)

result = retriever.query(
    query="Customer is in military",
    entitlement='agent_support',
    org_id='org_123',
    top_k=5,
    verbose=True
)

print("\nFinal Results:")
for i, doc in enumerate(result['documents'], 1):
    print(f"  {i}. {doc['document_name']} (score: {doc['score']:.4f})")

## Test 2: Multi-Turn Conversation with Query Expansion

In [ ]:
print("="*60)
print("TEST 2: Multi-Turn Conversation")
print("="*60)

# Create session
session_id = retriever.create_session(
    user_id='agent_001',
    entitlement='agent_support',
    org_id='org_123'
)

# Turn 1
print("\n" + "-"*50)
print("TURN 1")
print("-"*50)
r1 = retriever.query_with_session(
    session_id=session_id,
    query="How do I cancel a booking?",
    verbose=True
)
print(f"Top result: {r1['documents'][0]['document_name']}")

# Turn 2
print("\n" + "-"*50)
print("TURN 2")
print("-"*50)
r2 = retriever.query_with_session(
    session_id=session_id,
    query="What about the refund?",
    verbose=True
)
print(f"Top result: {r2['documents'][0]['document_name']}")

# Turn 3 - Ambiguous query (history helps)
print("\n" + "-"*50)
print("TURN 3 (Ambiguous - history provides context)")
print("-"*50)
r3 = retriever.query_with_session(
    session_id=session_id,
    query="What documents do I need?",
    verbose=True
)
print(f"Top result: {r3['documents'][0]['document_name']}")
print(f"\nExpanded query was: {r3['expanded_query']}")

## Test 3: Compare Expansion Modes

In [ ]:
print("="*60)
print("TEST 3: Compare Query Expansion Modes")
print("="*60)

test_query = "What documents do I need?"
test_history = [
    {'query': 'How do I cancel a booking?'},
    {'query': 'What is the refund policy?'}
]

modes = ['prepend', 'append', 'natural', 'keywords']

for mode in modes:
    expanded = retriever.expand_query(test_query, test_history, expansion_mode=mode)
    print(f"\n{mode.upper()}:")
    print(f"  {expanded}")
    
    result = retriever.query(
        query=test_query,
        entitlement='agent_support',
        org_id='org_123',
        conversation_history=test_history,
        expansion_mode=mode,
        top_k=3
    )
    
    print(f"  Results: {[d['document_name'] for d in result['documents']]}")

## Test 4: With vs Without History

In [ ]:
print("="*60)
print("TEST 4: Same Query With vs Without History")
print("="*60)

ambiguous_query = "What documents do I need?"

# Without history
print("\n--- WITHOUT HISTORY ---")
result_no_history = retriever.query(
    query=ambiguous_query,
    entitlement='agent_support',
    org_id='org_123',
    conversation_history=None,
    top_k=3
)
print(f"Query: {ambiguous_query}")
for i, doc in enumerate(result_no_history['documents'], 1):
    print(f"  {i}. {doc['document_name']}")

# With cancellation history
print("\n--- WITH CANCELLATION HISTORY ---")
cancel_history = [
    {'query': 'How do I cancel a booking?'},
    {'query': 'What is the refund policy?'}
]
result_cancel = retriever.query(
    query=ambiguous_query,
    entitlement='agent_support',
    org_id='org_123',
    conversation_history=cancel_history,
    top_k=3
)
print(f"Expanded: {result_cancel['expanded_query']}")
for i, doc in enumerate(result_cancel['documents'], 1):
    print(f"  {i}. {doc['document_name']}")

# With military history
print("\n--- WITH MILITARY HISTORY ---")
military_history = [
    {'query': 'Customer is in military'},
    {'query': 'Military discount eligibility'}
]
result_military = retriever.query(
    query=ambiguous_query,
    entitlement='agent_support',
    org_id='org_123',
    conversation_history=military_history,
    top_k=3
)
print(f"Expanded: {result_military['expanded_query']}")
for i, doc in enumerate(result_military['documents'], 1):
    print(f"  {i}. {doc['document_name']}")

## Summary

### How Query Expansion Works:

```
History:
  Turn 1: "How do I cancel a booking?"
  Turn 2: "What is the refund policy?"

Current Query: "What documents do I need?"

Expanded Query (prepend mode):
  "How do I cancel a booking?. What is the refund policy?. What documents do I need?"

→ Embedding now captures context: "documents" means cancellation/refund documents
```

### Expansion Modes:

| Mode | Output | Best For |
|------|--------|----------|
| `prepend` | `history. history. query` | General use |
| `append` | `query (context: history)` | When query is primary |
| `natural` | `Regarding history: query` | Natural phrasing |
| `keywords` | `history history query` | Keyword matching |

### Usage:

```python
# With automatic session tracking
result = retriever.query_with_session(
    session_id=session_id,
    query="What documents do I need?",
    expansion_mode='prepend'  # or 'natural', 'keywords', 'append'
)

# With manual history
result = retriever.query(
    query="What documents do I need?",
    entitlement='agent_support',
    conversation_history=[
        {'query': 'How do I cancel?'},
        {'query': 'Refund policy?'}
    ],
    expansion_mode='prepend'
)
```

In [ ]:
print("="*60)
print("NOTEBOOK COMPLETE")
print("="*60)
print("\nEmbedding-only retriever with query expansion is ready!")
print("No re-ranker needed.")